<a href="https://colab.research.google.com/github/ninobronw/BPALM-TOOL-IMPLEMENTATION/blob/main/Full_Code_NLP_RAG_Project_Notebook_version_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to understand issues like information overload and develop intelligent systems that can assist in quick decision-making.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and treatments.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries are installed.

In [52]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==1.24.4 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 45.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries are installed.

In [24]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.66 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 70.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 203.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 226.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 193.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
langgraph 1.2.1 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.42.1 which

## Question Answering using LLM

#### Downloading and Loading the model

In [21]:
from huggingface_hub import hf_hub_download

model_id = "TheBloke/Mistral-7B-Instruct-v0.1-GGUF"
model_basename = "mistral-7b-instruct-v0.1.Q4_K_M.gguf"

# Download the model from Hugging Face Hub
model_path = hf_hub_download(
    repo_id=model_id,
    filename=model_basename
)

In [25]:
import os

if os.path.exists(model_path):
    file_size_bytes = os.path.getsize(model_path)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f"The GGUF file exists at: {model_path}")
    print(f"File size: {file_size_mb:.2f} MB")
    if file_size_bytes == 0:
        print("Warning: The downloaded GGUF file has a size of 0 bytes, which indicates a potential issue with the download.")
else:
    print(f"Error: The GGUF file was not found at: {model_path}")

The GGUF file exists at: /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.1-GGUF/snapshots/731a9fc8f06f5f5e2db8a0cf9d256197eb6e05d1/mistral-7b-instruct-v0.1.Q4_K_M.gguf
File size: 4166.07 MB




```
# Ce texte est au format code
```

#### Initialize the LLM

In [30]:
from llama_cpp import Llama

# Initialize the LLM
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,  # -1 means use GPU if available, change to 0 for CPU-only
    n_ctx=2048,       # context window size
    verbose=True      # Set to True to avoid UnsupportedOperation: fileno error in Colab
)

print("✓ LLM initialized successfully!")

✓ LLM initialized successfully!


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [13]:
def response(query, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    """Generate a response from the LLM."""
    model_output = llm(
        prompt=query,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [31]:
query1 = "What is the protocol for managing sepsis in a critical care unit?"
print(response(f"[INST] {query1} [/INST]", max_tokens=256))

 Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit. The protocol for managing sepsis may vary depending on the specific hospital or healthcare facility, but generally, it involves the following steps:

1. Early recognition: Healthcare providers should be trained to recognize the signs and symptoms of sepsis and initiate treatment as soon as possible. This includes monitoring vital signs, blood cultures, and laboratory tests to identify infection and inflammation.
2. Source control: The source of infection should be identified and treated as soon as possible. This may involve removing infected tissue or drainage, administering antibiotics, or treating underlying conditions such as pneumonia or urinary tract infections.
3. Fluid resuscitation: Sepsis can cause fluid loss and dehydration, which can lead to organ failure. Healthcare providers should administer fluids and electrolytes to restore normal fluid balance and maintain or

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [32]:
query2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(f"[INST] {query2} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 The common symptoms of appendicitis include abdominal pain, fever, nausea, vomiting, diarrhea, and loss of appetite. In some cases, there may be no symptoms at all. 

Unfortunately, appendicitis cannot be cured via medicine alone. Surgery is the only effective treatment for this condition. The surgical procedure involves removing the appendix through a small incision in the abdomen. This procedure is usually performed under general anesthesia and takes about one to two hours. After surgery, patients are typically able to return home within a few days and resume normal activities within a week or so.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes?

In [33]:
query3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes?"
print(response(f"[INST] {query3} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 Sudden patchy hair loss can be caused by various factors such as stress, hormonal imbalances, autoimmune disorders, nutritional deficiencies, or certain medications. Here are some effective treatments or solutions for addressing sudden patchy hair loss:

1. Topical Treatments: Over-the-counter topical creams and serums containing minoxidil or finasteride can help stimulate hair growth and reduce hair fall.

2. Prescription Medications: Prescription medications such as corticosteroids, immunosuppressants, or anti-inflammatory drugs may be prescribed to treat the underlying cause of the hair loss.

3. Nutritional Supplements: Nutritional supplements containing biotin, iron, zinc, and other essential nutrients can help promote healthy hair growth.

4. Hair Transplant: In severe cases of sudden patchy hair loss, a hair transplant may be recommended to restore the lost hair.

5. Stress Management: Chronic stress can lead to hair loss. Practicing stress-reducing techniques such as meditatio

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [34]:
query4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(f"[INST] {query4} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 Treatment for a physical injury to brain tissue can vary depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Rest: It is important to give the brain time to heal by resting as much as possible. This may involve bed rest, limiting physical activity, and avoiding activities that require cognitive function such as reading or watching TV.

2. Medication: Depending on the severity of the injury, medication may be prescribed to manage pain, reduce inflammation, and prevent seizures.

3. Rehabilitation: Physical therapy, occupational therapy, and speech therapy can help improve physical function, mobility, and cognitive abilities. These therapies may involve exercises, stretches, and other techniques to promote healing and recovery.

4. Surgery: In some cases, surgery may be necessary to remove damaged tissue or relieve pressure on the brain. However, this is typically a last resort and is only recommended in severe cases where o

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [35]:
query5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(f"[INST] {query5} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 If someone has fractured their leg during a hiking trip, it is important to seek medical attention as soon as possible. Here are some necessary precautions and treatment steps that should be taken:

1. Immobilize the leg: Use a splint or cast to keep the leg still and prevent further damage. This will also help with healing.

2. Reduce pain: Over-the-counter pain relievers such as ibuprofen or acetaminophen can help manage pain. Prescription pain medication may be necessary in severe cases.

3. Rest: The leg should be kept still for several weeks to allow the bone to heal. This means avoiding weight-bearing activities such as walking or hiking.

4. Physical therapy: Once the bone has healed, physical therapy can help restore strength and flexibility to the leg. This is important for a full recovery.

5. Follow up with a doctor: Regular checkups with a doctor are necessary to monitor progress and ensure that the bone is healing properly.

6. Consider surgery: In some cases, surgery may

## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [36]:
query1 = "What is the protocol for managing sepsis in a critical care unit?"
print(response(f"[INST] {query1} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit. The protocol for managing sepsis may vary depending on the specific hospital or healthcare facility, but generally, it involves the following steps:

1. Early recognition: Healthcare providers should be trained to recognize the signs and symptoms of sepsis and initiate treatment as soon as possible. This includes monitoring vital signs, blood cultures, and laboratory tests to identify infection and inflammation.
2. Source control: The source of infection should be identified and treated as soon as possible. This may involve removing infected tissue or drainage, administering antibiotics, or treating underlying conditions such as pneumonia or urinary tract infections.
3. Fluid resuscitation: Sepsis can cause fluid loss and dehydration, which can lead to organ failure. Healthcare providers should administer fluids and electrolytes to restore normal fluid balance and maintain or

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [37]:
query2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(f"[INST] {query2} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 The common symptoms of appendicitis include abdominal pain, fever, nausea, vomiting, diarrhea, and loss of appetite. In some cases, there may be no symptoms at all. 

Unfortunately, appendicitis cannot be cured via medicine alone. Surgery is the only effective treatment for this condition. The surgical procedure involves removing the appendix through a small incision in the abdomen. This procedure is usually performed under general anesthesia and takes about one to two hours. After surgery, patients are typically able to return home within a few days and resume normal activities within a week or so.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes?

In [38]:
query3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes?"
print(response(f"[INST] {query3} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 Sudden patchy hair loss can be caused by various factors such as stress, hormonal imbalances, autoimmune disorders, nutritional deficiencies, or certain medications. Here are some effective treatments or solutions for addressing sudden patchy hair loss:

1. Topical Treatments: Over-the-counter topical creams and serums containing minoxidil or finasteride can help stimulate hair growth and reduce hair fall.

2. Prescription Medications: Prescription medications such as corticosteroids, immunosuppressants, or anti-inflammatory drugs may be prescribed to treat the underlying cause of the hair loss.

3. Nutritional Supplements: Nutritional supplements containing biotin, iron, zinc, and other essential nutrients can help promote healthy hair growth.

4. Hair Transplant: In severe cases of sudden patchy hair loss, a hair transplant may be recommended to restore the lost hair.

5. Stress Management: Chronic stress can lead to hair loss. Practicing stress-reducing techniques such as meditatio

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [39]:
query4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(f"[INST] {query4} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 Treatment for a physical injury to brain tissue can vary depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Rest: It is important to give the brain time to heal by resting as much as possible. This may involve bed rest, limiting physical activity, and avoiding activities that require cognitive function such as reading or watching TV.

2. Medication: Depending on the severity of the injury, medication may be prescribed to manage pain, reduce inflammation, and prevent seizures.

3. Rehabilitation: Physical therapy, occupational therapy, and speech therapy can help improve physical function, mobility, and cognitive abilities. These therapies may involve exercises, stretches, and other techniques to promote healing and recovery.

4. Surgery: In some cases, surgery may be necessary to remove damaged tissue or relieve pressure on the brain. However, this is typically a last resort and is only recommended in severe cases where o

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [40]:
query5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(f"[INST] {query5} [/INST]", max_tokens=256))

Llama.generate: prefix-match hit


 If someone has fractured their leg during a hiking trip, it is important to seek medical attention as soon as possible. Here are some necessary precautions and treatment steps that should be taken:

1. Immobilize the leg: Use a splint or cast to keep the leg still and prevent further damage. This will also help with healing.

2. Reduce pain: Over-the-counter pain relievers such as ibuprofen or acetaminophen can help manage pain. Prescription pain medication may be necessary in severe cases.

3. Rest: The leg should be kept still for several weeks to allow the bone to heal. This means avoiding weight-bearing activities such as walking or hiking.

4. Physical therapy: Once the bone has healed, physical therapy can help restore strength and flexibility to the leg. This is important for a full recovery.

5. Follow up with a doctor: Regular checkups with a doctor are necessary to monitor progress and ensure that the bone is healing properly.

6. Consider surgery: In some cases, surgery may

## Data Preparation for RAG

### Loading the Data

In [45]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings

import os
loader = PyMuPDFLoader("medical_diagnosis_manual.pdf")
documents = loader.load()
print(f"Current working directory: {os.getcwd()}")
print("Files in the current directory:")
for item in os.listdir(os.getcwd()):
    print(item)

Current working directory: /content
Files in the current directory:
.config
medical_diagnosis_manual.pdf
sample_data


### Data Overview

#### Checking the first 5 pages

#### Checking the number of pages

In [53]:
print(f"Total number of documents (pages): {len(documents)}")

print("\nFirst 5 documents (pages) content summary:")
for i, doc in enumerate(documents[:5]):
    print(f"\n--- Document {i+1} ---")
    print(f"Metadata: {doc.metadata}")
    print(f"Content preview (first 200 chars):\n{doc.page_content[:200]}...")

Total number of documents (pages): 4114

First 5 documents (pages) content summary:

--- Document 1 ---
Metadata: {'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': 'medical_diagnosis_manual.pdf', 'file_path': 'medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-02T21:24:41+00:00', 'trapped': '', 'modDate': 'D:20260602212441Z', 'creationDate': 'D:20120615054440Z', 'page': 0}
Content preview (first 200 chars):
jlabena@gmail.com
7IJP65QWB1
This file is meant for personal use by jlabena@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action....

--- Document 2 ---
Metadata: {'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-

### Data Chunking

In [43]:
# Split the loaded pages into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # max size of a chunk (in characters)
    chunk_overlap=200     # overlap between two consecutive chunks
)

# 'data' = the list of Documents from the "Loading the Data" step
document_chunks = text_splitter.split_documents(documents)

# Checks
print(f"Number of pages loaded : {len(documents)}")
print(f"Number of chunks created : {len(document_chunks)}")
print("\nExample chunk:\n")
print(document_chunks[10].page_content[:500])

Number of pages loaded : 4114
Number of chunks created : 17956

Example chunk:

322
Chapter 28. Hepatitis    ..................................................................................................................................................................
333
Chapter 29. Vascular Disorders of the Liver    .....................................................................................................................
341
Chapter 30. Liver Masses & Granulomas    ..............................................................................................


### Embedding

In [55]:
# Create the embedding model (turns text chunks into numeric vectors)
embedding_model = SentenceTransformerEmbeddings(
    model_name="thenlper/gte-small"
)

# Quick check: embed one short text and look at the vector size
sample_vector = embedding_model.embed_query("What is sepsis?")
print(f"Embedding model loaded : thenlper/gte-small")
print(f"Vector dimension       : {len(sample_vector)}")
print(f"First 5 values         : {sample_vector[:5]}")

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

### Vector Database

In [47]:
vectordb = Chroma.from_documents(
    documents=document_chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db" # Persist the database to disk
)

NameError: name 'embedding_model' is not defined

### Retriever

In [48]:
retriever = vectordb.as_retriever(
    search_type="similarity", # Use similarity search
    search_kwargs={"k": 3}    # Return top 3 most similar documents
)

NameError: name 'vectordb' is not defined

### System and User Prompt Template

In [49]:
qna_system_message = """You are a helpful medical AI assistant. You answer questions based on the provided medical context."""
qna_user_message_template = """Here is the medical context: {context}

Answer the following question: {question}"""

In [50]:
def generate_rag_response(user_input, k=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response_output = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )

        # Extract and print the model's response
        response_output = response_output['choices'][0]['text'].strip()
    except Exception as e:
        response_output = f'Sorry, I encountered the following error: \n {e}'

    return response_output

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [51]:
query1 = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(query1, max_tokens=256))

NameError: name 'retriever' is not defined

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(query2, max_tokens=256))

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes?

In [ ]:
query3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes?"
print(generate_rag_response(query3, max_tokens=256))

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(query4, max_tokens=256))

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(query5, max_tokens=256))

### Fine-tuning

Fine-tuning was not implemented in this project. It was considered as a fourth possible approach, but RAG was chosen as the more appropriate solution for this use case.

**What fine-tuning is**: Fine-tuning re-trains the weights of the base model on a domain-specific dataset, so the new knowledge and behavior become embedded directly in the model itself.